In [1]:
import os
os.makedirs("/kaggle/temp", exist_ok=True)
!cp /kaggle/input/notebooks/fahimratul/thesis-of-cnn/damage.h5 /kaggle/temp/damage.h5
print("done, size:")
!ls -lh /kaggle/temp/damage.h5

done, size:
-rw-r--r-- 1 root root 3.5G Jul 25 09:02 /kaggle/temp/damage.h5


In [3]:
%%writefile /kaggle/working/02_train_damage.py

import argparse
import io
import json
import time
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


# ==================================================== Dataset

class H5PatchDataset(Dataset):
    """
    Reads PNG bytes from HDF5 and decodes them.

    Important: an h5py file handle cannot be shared across multiprocessing.
    So each worker opens its own handle on first access (lazy open).
    """

    def __init__(self, h5_path, split, train=True, normalize="simple"):
        self.h5_path = str(h5_path)
        self.split = split
        self.train = train
        self.normalize = normalize
        self._h5 = None

        with h5py.File(self.h5_path, "r") as f:
            g = f[split]
            self.length = g["label"].shape[0]
            self.labels = g["label"][:].astype(np.int64)
            self.classes = json.loads(g.attrs["classes"])

    def _ensure_open(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5[self.split]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        g = self._ensure_open()
        png = g["png"][idx].tobytes()
        arr = np.asarray(Image.open(io.BytesIO(png)).convert("RGB"),
                         dtype=np.float32) / 255.0

        if self.train:
            if np.random.rand() < 0.5:
                arr = arr[:, ::-1]
            if np.random.rand() < 0.5:
                arr = arr[::-1, :]
            k = np.random.randint(4)
            if k:
                arr = np.rot90(arr, k)
            if np.random.rand() < 0.5:
                arr = np.clip(arr * np.random.uniform(0.8, 1.2), 0, 1)

        if self.normalize == "imagenet":
            arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
        else:
            arr = (arr - 0.5) / 0.5

        arr = np.ascontiguousarray(arr.transpose(2, 0, 1))
        return torch.from_numpy(arr), int(self.labels[idx])


# ==================================================== models

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(cout)

    def forward(self, x):
        return F.max_pool2d(F.relu(self.bn(self.conv(x))), 2)


class SimpleCNN(nn.Module):
    """128 -> 64 -> 32 -> 16 -> 8 -> 4, channels 32→64→128→256→512"""

    def __init__(self, num_classes=4, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32), ConvBlock(32, 64), ConvBlock(64, 128),
            ConvBlock(128, 256), ConvBlock(256, 512),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))


def build_resnet(num_classes=4, freeze_early=True):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    if freeze_early:
        for name, p in m.named_parameters():
            if name.startswith(("conv1", "bn1", "layer1", "layer2")):
                p.requires_grad = False
    m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, num_classes))
    return m


# ==================================================== evaluation

@torch.no_grad()
def evaluate(model, loader, device, criterion, n_cls):
    model.eval()
    preds, targets, losses = [], [], []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(x)
            losses.append(criterion(logits, y).item())
        preds.append(logits.argmax(1).cpu())
        targets.append(y.cpu())
    preds = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    return {
        "loss": float(np.mean(losses)),
        "acc": float((preds == targets).mean()),
        "macro_f1": float(f1_score(targets, preds, average="macro", zero_division=0)),
        "per_class": f1_score(targets, preds, average=None,
                              labels=list(range(n_cls)), zero_division=0),
        "preds": preds, "targets": targets,
    }


# ==================================================== Training

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--h5", default="data/damage.h5")
    ap.add_argument("--out_dir", default="./runs_full")
    ap.add_argument("--model", choices=["cnn", "resnet"], default="cnn")
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--batch_size", type=int, default=128)
    ap.add_argument("--lr", type=float, default=None)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--val_subset", type=int, default=0,
                    help="how many val samples per epoch (0 = all). Saves time on a large val set")
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.backends.cudnn.benchmark = True

    norm = "imagenet" if args.model == "resnet" else "simple"
    train_ds = H5PatchDataset(args.h5, "train", train=True, normalize=norm)
    val_ds = H5PatchDataset(args.h5, "val", train=False, normalize=norm)
    classes = train_ds.classes
    n_cls = len(classes)
    print(f"Device: {device} | model: {args.model}")
    print(f"Train {len(train_ds):,} | Val {len(val_ds):,} | classes {classes}")

    counts = np.bincount(train_ds.labels, minlength=n_cls).astype(np.float64)
    print("Train distribution:", dict(zip(classes, counts.astype(int).tolist())))
    sample_w = (1.0 / np.maximum(counts, 1))[train_ds.labels]
    sampler = WeightedRandomSampler(torch.as_tensor(sample_w, dtype=torch.double),
                                    num_samples=len(sample_w), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, sampler=sampler,
                              num_workers=args.workers, pin_memory=True,
                              drop_last=True, persistent_workers=args.workers > 0)

    val_indices = None
    if args.val_subset and args.val_subset < len(val_ds):
        rng = np.random.RandomState(0)
        val_indices = rng.choice(len(val_ds), args.val_subset, replace=False)
        val_loader = DataLoader(torch.utils.data.Subset(val_ds, val_indices.tolist()),
                                batch_size=args.batch_size * 2, shuffle=False,
                                num_workers=args.workers, pin_memory=True)
        print(f"(val subset {args.val_subset:,} per epoch — the full set runs at the end)")
    else:
        val_loader = DataLoader(val_ds, batch_size=args.batch_size * 2, shuffle=False,
                                num_workers=args.workers, pin_memory=True)

    model = (SimpleCNN(n_cls) if args.model == "cnn" else build_resnet(n_cls)).to(device)
    lr = args.lr or (1e-3 if args.model == "cnn" else 1e-4)
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,} | LR {lr}")

    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

    ckpt_path = out_dir / f"last_{args.model}.pt"
    best_path = out_dir / f"best_{args.model}.pt"
    start_epoch, best_f1, history = 1, 0.0, []

    if args.resume and ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"])
        scaler.load_state_dict(ck["scaler"])
        start_epoch = ck["epoch"] + 1
        best_f1 = ck["best_f1"]
        history = ck.get("history", [])
        print(f">>> Resume: starting from epoch {start_epoch} (best F1 {best_f1:.4f})")

    for epoch in range(start_epoch, args.epochs + 1):
        model.train()
        t0, running = time.time(), []
        for step, (x, y) in enumerate(train_loader, 1):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            running.append(loss.item())
            if step % 200 == 0:
                done = step * args.batch_size
                rate = done / (time.time() - t0)
                print(f"  ep{epoch} {step}/{len(train_loader)} "
                      f"loss {np.mean(running[-200:]):.4f} | {rate:.0f} img/s")

        scheduler.step()
        m = evaluate(model, val_loader, device, criterion, n_cls)
        mins = (time.time() - t0) / 60
        print(f"[Epoch {epoch:02d}] {mins:.1f} min | train {np.mean(running):.4f} | "
              f"val {m['loss']:.4f} | acc {m['acc']:.4f} | macro F1 {m['macro_f1']:.4f}")
        for name, f1 in zip(classes, m["per_class"]):
            print(f"    {name:15s} F1 = {f1:.4f}")

        history.append({"epoch": epoch, "train_loss": float(np.mean(running)),
                        "val_loss": m["loss"], "acc": m["acc"],
                        "macro_f1": m["macro_f1"],
                        "per_class": m["per_class"].tolist()})

        # save every epoch — no work lost even if Colab disconnects
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                    "epoch": epoch, "best_f1": best_f1, "history": history,
                    "arch": args.model, "classes": classes}, ckpt_path)

        if m["macro_f1"] > best_f1:
            best_f1 = m["macro_f1"]
            torch.save({"model": model.state_dict(), "epoch": epoch,
                        "macro_f1": best_f1, "arch": args.model,
                        "classes": classes}, best_path)
            print(f"    -> best model saved (macro F1 {best_f1:.4f})")
        print()

    # ---------- final: on the full val set ----------
    ck = torch.load(best_path, map_location=device)
    model.load_state_dict(ck["model"])
    full_val = DataLoader(val_ds, batch_size=args.batch_size * 2, shuffle=False,
                          num_workers=args.workers, pin_memory=True)
    final = evaluate(model, full_val, device, criterion, n_cls)

    print("=" * 62)
    print(f"{args.model.upper()} — best model (epoch {ck['epoch']}) | full val set")
    print("=" * 62)
    print(classification_report(final["targets"], final["preds"],
                                target_names=classes, digits=4, zero_division=0))
    print("Confusion matrix (rows=actual, cols=predicted):")
    print(confusion_matrix(final["targets"], final["preds"]))

    with open(out_dir / f"history_{args.model}.json", "w") as f:
        json.dump({"history": history, "best_macro_f1": best_f1,
                   "final_macro_f1": final["macro_f1"]}, f, indent=2)


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/02_train_damage.py


In [4]:
!python /kaggle/working/02_train_damage.py \
    --h5 /kaggle/temp/damage.h5 \
    --out_dir /kaggle/working/runs \
    --model cnn --epochs 25 \
    --batch_size 128 --workers 2 --val_subset 20000

Device: cuda | model: cnn
Train 155,765 | Val 60,041 | classes ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
Train distribution: {'no-damage': 100000, 'minor-damage': 19997, 'major-damage': 17083, 'destroyed': 18685}
(val subset 20,000 per epoch — the full set runs at the end)
Parameters: 1,701,924 | LR 0.001
  ep1 200/1216 loss 1.2050 | 648 img/s
  ep1 400/1216 loss 1.0993 | 786 img/s
  ep1 600/1216 loss 1.0594 | 852 img/s
  ep1 800/1216 loss 1.0207 | 888 img/s
  ep1 1000/1216 loss 0.9953 | 911 img/s
  ep1 1200/1216 loss 0.9787 | 931 img/s
[Epoch 01] 3.1 min | train 1.0585 | val 1.0527 | acc 0.5786 | macro F1 0.4924
    no-damage       F1 = 0.6967
    minor-damage    F1 = 0.3444
    major-damage    F1 = 0.3294
    destroyed       F1 = 0.5993
    -> best model saved (macro F1 0.4924)

  ep2 200/1216 loss 0.9459 | 1014 img/s
  ep2 400/1216 loss 0.9400 | 1020 img/s
  ep2 600/1216 loss 0.9202 | 1025 img/s
  ep2 800/1216 loss 0.9117 | 1024 img/s
  ep2 1000/1216 loss 0.9069 | 1